# Acquire and Prepare data 

In [26]:
# For data manipulation
import pandas as pd
import numpy as np

# For fetching data from APIs
import requests



## I will be accessing the OpenAlex API
### Attributes of interest:

display_name - this is the title of the paper and is where most of our information will be extracted from. 

publication_year (int) - this is the year the paper was published.

cited_total (int) - I believe this is the number of other published papers that cite this one. Will be useful in determining popularity or relavance of research fields 

language - I will be looking at publications made in english

institution -> display_name - the instution of the first listed author on the research paper 

country - where the paper was published 

domain - which field this paper falls within

field - which field does this paper fall in

subfield - subfield paper falls in

cited_yearly - how many other papers cited this one year by year. 

## Retrieve the data from OpenAlex

### This takes a considerable amount of time as I am pinging the API 10k times collecting 200 data point on each iteration. I try not to run this often as it can take ~3h or more to complete. The data collection is also alwaqys random meaning every time I run this I will be getting a different set of 200k data points. 

In [27]:

req_count = 1
page = 1
data = []

for i in range(10000):
    response = requests.get(f'https://api.openalex.org/works?per-page=200&page={page}')
    if response.status_code == 200:
        print(f'Request {req_count}: Successful!')
        res = response.json()
        if not res['results']:
            break  
        
        for item in res['results']:
            title = item.get('display_name')
            pub_year = item.get('publication_year')
            language = item.get('language')
            institution = None
            country = None
            domain = None
            field = None
            subfield = None
            cited_yearly = []
            cited_total = item.get('cited_by_count', 0)

            if item.get('authorships') and item['authorships'][0].get('institutions'):
                institution = item['authorships'][0]['institutions'][0].get('display_name')

            if item.get('authorships') and item['authorships'][0].get('countries'):
                country = item['authorships'][0]['countries'][0]

            if item.get('topics') and item['topics'][0].get('domain'):
                domain = item['topics'][0]['domain'].get('display_name')

            if item.get('topics') and item['topics'][0].get('field'):
                field = item['topics'][0]['field'].get('display_name')

            if item.get('primary_topic') and item['primary_topic']['subfield']:
                subfield = item['primary_topic']['subfield'].get('display_name')

            if item.get('counts_by_year'):
                cited_yearly = [(entry["year"], entry["cited_by_count"]) for entry in item['counts_by_year']]

            data.append({
                'title': title,
                'publication_year': pub_year,
                'cited_total': cited_total,
                'language': language,
                'institution': institution,
                'country': country,
                'domain': domain,
                'field': field,
                'subfield': subfield,
                'cited_yearly': cited_yearly,
            })

        # Pagination handling
        if req_count % 200 == 0:
            page += 1
        req_count += 1
    else:
        print(f'Request {req_count}: Failed')
        break  # Exit the loop if the request failed

if data:
    print(data[0])
else:
    print("No data retrieved.")


Request 1: Successful!
Request 2: Successful!
Request 3: Successful!
Request 4: Successful!
Request 5: Successful!
Request 6: Successful!
Request 7: Successful!
Request 8: Successful!
Request 9: Successful!
Request 10: Successful!
Request 11: Successful!
Request 12: Successful!
Request 13: Successful!
Request 14: Successful!
Request 15: Successful!
Request 16: Successful!
Request 17: Successful!
Request 18: Successful!
Request 19: Successful!
Request 20: Successful!
Request 21: Successful!
Request 22: Successful!
Request 23: Successful!
Request 24: Successful!
Request 25: Successful!
Request 26: Successful!
Request 27: Successful!
Request 28: Successful!
Request 29: Successful!
Request 30: Successful!
Request 31: Successful!
Request 32: Successful!
Request 33: Successful!
Request 34: Successful!
Request 35: Successful!
Request 36: Successful!
Request 37: Successful!
Request 38: Successful!
Request 39: Successful!
Request 40: Successful!
Request 41: Successful!
Request 42: Successful!
R

In [31]:
print(data[:2])

[{'title': 'PROTEIN MEASUREMENT WITH THE FOLIN PHENOL REAGENT', 'publication_year': 1951, 'cited_total': 304672, 'language': 'en', 'institution': 'Washington University in St. Louis', 'country': 'US', 'domain': 'Life Sciences', 'field': 'Biochemistry, Genetics and Molecular Biology', 'subfield': 'Molecular Biology', 'cited_yearly': [(2024, 288), (2023, 2714), (2022, 2963), (2021, 3113), (2020, 3241), (2019, 3016), (2018, 3078), (2017, 3086), (2016, 3585), (2015, 4220), (2014, 4683), (2013, 4727), (2012, 4678)]}, {'title': 'Cleavage of Structural Proteins during the Assembly of the Head of Bacteriophage T4', 'publication_year': 1970, 'cited_total': 233554, 'language': 'en', 'institution': 'MRC Laboratory of Molecular Biology', 'country': 'GB', 'domain': 'Physical Sciences', 'field': 'Environmental Science', 'subfield': 'Ecology', 'cited_yearly': [(2024, 202), (2023, 1805), (2022, 2170), (2021, 2353), (2020, 2531), (2019, 2435), (2018, 2679), (2017, 2755), (2016, 3093), (2015, 3546), (20

In [29]:
df = pd.DataFrame(data)
print(df.head(5))

# Save the DataFrame to a CSV file so the data can be grabbed dirtectly from the csv
df.to_csv('research_paper_data.csv', index=False)

                                               title  publication_year  \
0  PROTEIN MEASUREMENT WITH THE FOLIN PHENOL REAGENT              1951   
1  Cleavage of Structural Proteins during the Ass...              1970   
2  A rapid and sensitive method for the quantitat...              1976   
3     Generalized Gradient Approximation Made Simple              1996   
4  Analysis of Relative Gene Expression Data Usin...              2001   

   cited_total language                          institution country  \
0       304672       en   Washington University in St. Louis      US   
1       233554       en  MRC Laboratory of Molecular Biology      GB   
2       186233       en                University of Georgia      US   
3       156113       en                    Tulane University      US   
4       142953       en                                 None      US   

              domain                                         field  \
0      Life Sciences  Biochemistry, Genetics and Mol